# Load Dataset

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import re
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')




[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Moses\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Moses\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Moses\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Moses\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Moses\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [2]:
dataset = load_dataset("stanfordnlp/imdb")
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
# print(train_df.shape)
# train_df.head()
print(train_df["label"].value_counts())

label
0    12500
1    12500
Name: count, dtype: int64


In [3]:
train_df["sentiment"] = train_df["label"].map({0:"negative", 1:"positive"})
test_df["sentiment"] = train_df["label"].map({0:"negative", 1:"positive"})
train_df.head()

,text,label,sentiment
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,negative
1,"""I Am Curious: Yellow"" is a risible and preten...",0,negative
2,If only to avoid making this type of film in t...,0,negative
3,This film was probably inspired by Godard's Ma...,0,negative
4,"Oh, brother...after hearing about this ridicul...",0,negative


In [4]:
# for i in range(3):
#     print(f"Review {i}")
#     print(train_df["text"][i][:100])
#     print(f"Sentiment : {train_df["sentiment"][i]}")
#     print("\n")

# Preprocessing

In [5]:
def lower_case(txt):
    txt = txt.lower()
    return txt

def remove_html_tags(txt):
    txt = re.sub(r"<[^>]*>", " ", txt)
    return txt

def remove_urls(txt):
    txt = re.sub(r"http[s]?://\S+", " ", txt)
    txt = re.sub(r"www\S+", "", txt)
    return txt

def remove_special_char(txt):
    txt = re.sub(r"[^\s\w]", "", txt)
    return txt

def contract(txt):
    txt = contractions.fix(txt)
    return txt

In [6]:
def clean_text(txt):
    lwr = lower_case(txt)
    rm_tag = remove_html_tags(lwr)
    rm_url = remove_urls(rm_tag)
    con = contract(rm_url)
    rm_sc = remove_special_char(con)

    return rm_sc

In [7]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [8]:
def tokenization(text):
    token = word_tokenize(text)
    return token


def stop_words(text):
    filter = []
    sw = set(stopwords.words('english'))
    for word in text:
        if word not in sw:
            filter.append(word)
    return filter


def lemmatization(text):
    lemma = []
    lemmatizer = WordNetLemmatizer()
    for word in text:
        lemma.append(lemmatizer.lemmatize(word))
    return lemma

In [9]:
def preprocessing(text):
    tokenize = tokenization(text)
    stop_word = stop_words(tokenize)
    lemmatize = lemmatization(stop_word)
    return " ".join(lemmatize)

In [ ]:
train_df["preprocessing"] = train_df["clean_text"].apply(preprocessing)
test_df["preprocessing"] = test_df["clean_text"].apply(preprocessing)

In [ ]:
train_df["preprocessing"][0]

'rented curiousyellow video store controversy surrounded first released 1967 also heard first seized yous custom ever tried enter country therefore fan film considered controversial really see plot centered around young swedish drama student named lena want learn everything life particular want focus attention making sort documentary average swede thought certain political issue vietnam war race issue united state asking politician ordinary denizen stockholm opinion politics sex drama teacher classmate married men kill curiousyellow 40 year ago considered pornographic really sex nudity scene far even shot like cheaply made porno countryman mind find shocking reality sex nudity major staple swedish cinema even ingmar bergman arguably answer good old boy john ford sex scene film commend filmmaker fact sex shown film shown artistic purpose rather shock people make money shown pornographic theater america curiousyellow good film anyone wanting study meat potato pun intended swedish cinema 

# Feature Engineering (TF-IDF Model)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
X_train = train_df["preprocessing"]
Y_train = train_df["sentiment"]

X_test = test_df["preprocessing"]
Y_test = test_df["sentiment"]

vectorizer = TfidfVectorizer()
train_x_vect = vectorizer.fit_transform(X_train)
# print(train_vect.shape)

test_x_vect = vectorizer.transform(X_test)

# Training  -TFIDF Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(train_x_vect, Y_train)

prediction = model.predict(test_x_vect)

text = ["The movie was really great haha", "i will rate this 3 out of 10 it really fuck the movie is not in a good terms"]
tes = vectorizer.transform(text)
prediction_tes = model.predict(tes)
prediction_tes


array(['positive', 'positive'], dtype=object)

In [ ]:
print(classification_report(Y_test, prediction))

              precision    recall  f1-score   support

    negative       0.88      0.88      0.88     12500
    positive       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



# Import Model (DistilBERT)

In [17]:
from transformers import pipeline

In [21]:
distilBert_model = pipeline('sentiment-analysis', model="omidroshani/imdb-sentiment-analysis")
response = distilBert_model("This movie is sucks")[0]["label"]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [22]:
result = []
sample = test_df.sample(n=100, random_state=42)
sample_text = sample["text"]
sample_label = sample["sentiment"]

for text in sample_text:
    text = text[:512]
    response = distilBert_model(text)[0]["label"]
    if response == "LABEL_0":
        response = "negative"
    else:
        response = "positive"
    result.append(response)


In [20]:
print(classification_report(sample_label, result))

              precision    recall  f1-score   support

    negative       0.81      0.89      0.85        47
    positive       0.90      0.81      0.85        53

    accuracy                           0.85       100
   macro avg       0.85      0.85      0.85       100
weighted avg       0.85      0.85      0.85       100



<h1> Download Models </h1>



In [23]:
import pickle 

tf_idf_model = model
pickle.dump(tf_idf_model, open("tf_idf_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))


distilBert_model.save_pretrained("./distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]